In [ ]:
## 🚀 AUTOMATED T5 TRAINING SCRIPT ON KAGGLE
## Before running: Upload dataset containing .py files to Kaggle

import os
import shutil
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==================================================================================
# EXPERIMENTAL CONFIGURATION (YOU ONLY NEED TO EDIT HERE)
# ==================================================================================
CONFIG = {
    "model_name": "t5-small",              # or "t5-base"
    "epochs": 100,                         # Number of epochs
    "batch_size": 128,                     # Batch size
    "train_samples": 1000,                 # Number of training samples (default in generate_data)
    "format": "10e-based",                 # Format: decimal, 10e-based, character, etc.
    "digits": [2, 5, 10, 15, 20, 25, 30],  # Digit lengths to test
    "num_runs": 5,                         # Number of runs with different seeds (1-5)
    "output_dir": "outputs/run_full_auto", # Output directory
    "seed": 42,                            # Starting seed
    "only_addition": True,                 # True = addition only, False = add + sub
    "max_seq_length": 512                  # Max sequence length
}
# ==================================================================================

def run_command(command):
    """Run shell command and check result"""
    print(f"⚡ Running: {command[:80]}...")
    exit_code = os.system(command)
    if exit_code != 0:
        print(f"❌ ERROR: Command failed with exit code {exit_code}")
        print(f"   Command: {command}")
        return False
    return True

print("=" * 80)
print("🚀 START AUTOMATED T5 TRAINING PROCESS")
print("=" * 80)

# ----------------------------------------------------------------------------------
# STEP 1: AUTOMATICALLY FIND AND COPY CODE FROM /kaggle/input
# ----------------------------------------------------------------------------------
print("\n[1/6] 📁 Copying Python files from input...")
source_root = "/kaggle/input"
dest_dir = "/kaggle/working"
py_files_count = 0

for root, dirs, files in os.walk(source_root):
    for file in files:
        if file.endswith(".py"):
            src_path = os.path.join(root, file)
            dst_path = os.path.join(dest_dir, file)
            shutil.copy(src_path, dst_path)
            print(f"   ✓ Copied: {file}")
            py_files_count += 1

if py_files_count == 0:
    raise FileNotFoundError("❌ CRITICAL: No .py files found! Please upload dataset code first.")

print(f"✅ Copied {py_files_count} Python files successfully\n")

# ----------------------------------------------------------------------------------
# STEP 2: INSTALL LIBRARIES
# ----------------------------------------------------------------------------------
print("[2/6] 📦 Installing required packages...")
if not run_command("pip install -q pytorch-lightning transformers num2words protobuf==3.20.3 libraft-cu12>=25.6.0 rmm-cu12>=25.6.0"):
    raise RuntimeError("❌ Failed to install packages!")
print("✅ Packages installed successfully\n")

# ----------------------------------------------------------------------------------
# STEP 3: CHECK REQUIRED FILES
# ----------------------------------------------------------------------------------
print("[3/6] 🔍 Checking required files...")
required_files = ["generate_data.py", "utils.py", "train.py", "dataset.py", "config.py"]
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    raise FileNotFoundError(f"❌ Missing files: {', '.join(missing_files)}")

print("✅ All required files found\n")

# ----------------------------------------------------------------------------------
# STEP 4: CREATE DIRECTORIES AND GENERATE DATA
# ----------------------------------------------------------------------------------
print("[4/6] 📊 Generating datasets...")
run_command("mkdir -p data outputs")

# Generate data with experiment mode (automatically for all digits)
gen_flags = "--only_addition" if CONFIG["only_addition"] else ""
gen_cmd = (
    f"python generate_data.py "
    f"--mode experiment "
    f"--base_path data/experiment "
    f"--seed {CONFIG['seed']} "
    f"--format {CONFIG['format']} "
    f"--digit_steps {' '.join(map(str, CONFIG['digits']))} "
    f"{gen_flags}"
)

if not run_command(gen_cmd):
    raise RuntimeError("❌ Data generation failed!")

print("✅ Data generated successfully\n")

# ----------------------------------------------------------------------------------
# STEP 5: TRAIN MODEL FOR ALL CONFIGURATIONS
# ----------------------------------------------------------------------------------
print(f"[5/6] 🏋️ Training model '{CONFIG['model_name']}'...")
print(f"   Digits to test: {CONFIG['digits']}")
print(f"   Runs per digit: {CONFIG['num_runs']}")
print(f"   Total trainings: {len(CONFIG['digits']) * CONFIG['num_runs']}\n")

training_count = 0
success_count = 0
failed_runs = []

for d in CONFIG['digits']:
    print(f"\n{'='*80}")
    print(f"🔢 DIGITS: {d}")
    print(f"{'='*80}")
    
    for run_id in range(1, CONFIG['num_runs'] + 1):  # 1 to 5 (or num_runs)
        training_count += 1
        print(f"\n[Run {run_id}/{CONFIG['num_runs']}] Training with {d} digits...")
        
        # Data filenames follow generate_data.py convention (train_1.json, val_1.json, test_1.json)
        train_file = f"data/experiment/digits_{d}/train_{run_id}.json"
        val_file = f"data/experiment/digits_{d}/val_{run_id}.json"
        test_file = f"data/experiment/digits_{d}/test_{run_id}.json"
        
        # Check if file exists
        if not all(os.path.exists(f) for f in [train_file, val_file, test_file]):
            print(f"   ⚠️  Skipping: Data files not found for digits={d}, run={run_id}")
            failed_runs.append(f"digits_{d}_run_{run_id} (missing data)")
            continue
        
        output_path = f"{CONFIG['output_dir']}/digits_{d}/run_{run_id}"
        
        train_cmd = (
            f"python train.py "
            f"--train_file {train_file} "
            f"--val_file {val_file} "
            f"--test_file {test_file} "
            f"--output_dir {output_path} "
            f"--model_name_or_path {CONFIG['model_name']} "
            f"--format {CONFIG['format']} "
            f"--train_size_log {CONFIG['train_samples']} "
            f"--sampling_strategy balanced "
            f"--epochs {CONFIG['epochs']} "
            f"--train_batch_size {CONFIG['batch_size']} "
            f"--val_batch_size {CONFIG['batch_size']} "
            f"--lr 3e-4 "
            f"--seed {CONFIG['seed'] + run_id} "
            f"--accelerator gpu "
            f"--devices 1 "
            f"--max_seq_length {CONFIG['max_seq_length']}"
        )
        
        if run_command(train_cmd):
            print(f"   ✅ Training completed successfully!")
            success_count += 1
        else:
            print(f"   ❌ Training failed!")
            failed_runs.append(f"digits_{d}_run_{run_id}")

print(f"\n{'='*80}")
print(f"📊 TRAINING SUMMARY")
print(f"{'='*80}")
print(f"Total trainings: {training_count}")
print(f"Successful: {success_count}")
print(f"Failed: {training_count - success_count}")
if failed_runs:
    print(f"\nFailed runs:")
    for run in failed_runs:
        print(f"   - {run}")
print(f"{'='*80}\n")

# ----------------------------------------------------------------------------------
# STEP 6: COMPRESS RESULTS
# ----------------------------------------------------------------------------------
print("[6/6] 📦 Compressing results...")

# Check if output directory exists
if not os.path.exists(CONFIG['output_dir']):
    print("⚠️  Warning: Output directory not found, creating empty archive")
    run_command(f"mkdir -p {CONFIG['output_dir']}")

zip_cmd = f"zip -r ket_qua_full.zip {CONFIG['output_dir']}"
if run_command(zip_cmd):
    print("✅ Results compressed successfully!")
    
    # Check file size
    if os.path.exists("ket_qua_full.zip"):
        size_mb = os.path.getsize("ket_qua_full.zip") / (1024 * 1024)
        print(f"   📦 Archive size: {size_mb:.2f} MB")
else:
    print("❌ Compression failed!")

print("\n" + "="*80)
print("🎉 ALL DONE! FILE 'ket_qua_full.zip' IS READY.")
print("="*80)
print(f"\n💡 Tips:")
print(f"   - Results are saved in: {CONFIG['output_dir']}")
print(f"   - Each run includes results.json and checkpoint models")
print(f"   - Check logs to inspect accuracy of each experiment")